# F1 Fantasy 2026 - Prediction Model

## Modelling Formula 1 Outcomes for Fantasy EV Projections (2026 Season)

Adapted from the 2024 model with key updates for the 2026 regulation era:
- **22 drivers / 11 teams** (Cadillac enters as 11th team, Sauber becomes Audi)
- **New power unit regulations**: 50/50 ICE/electric split, MGU-H removed, MGU-K tripled to 350kW
- **Active aerodynamics**: DRS replaced by Boost/Overtake modes
- **Updated Fantasy scoring**: Sprint DNF now -10 (was -20), price floor $3M
- **Regulation-reset uncertainty**: Historical data weighted to account for new era

## 2026 Season Overview

### Key Rule Changes
- Smaller, lighter cars (768kg min, 200mm shorter wheelbase)
- Active aero replaces DRS (Boost button + Overtake Mode)
- 50/50 power split: ~400kW ICE + ~350kW MGU-K
- 100% sustainable fuels
- No fastest lap bonus point in real F1 (abolished 2025), but Fantasy still awards fastest lap points

### 2026 Grid
| Team | Driver 1 | Driver 2 | PU |
|------|----------|----------|-----|
| McLaren | Norris | Piastri | Mercedes |
| Mercedes | Russell | Antonelli | Mercedes |
| Red Bull | Verstappen | Hadjar | Red Bull/Ford |
| Ferrari | Leclerc | Hamilton | Ferrari |
| Williams | Albon | Sainz | Mercedes |
| Racing Bulls | Lawson | Lindblad | Red Bull/Ford |
| Aston Martin | Alonso | Stroll | Honda |
| Haas | Bearman | Ocon | Ferrari |
| Audi | Hulkenberg | Bortoleto | Audi |
| Alpine | Gasly | Colapinto | Mercedes |
| Cadillac | Perez | Bottas | Ferrari |

### Sprint Weekends (6 total)
China, Miami, Canada, Great Britain, Netherlands, Singapore

In [ ]:
%pip install -q unidecode gurobipy pulp pandas seaborn scipy scikit-learn matplotlib

In [ ]:
import numpy as np
import pandas as pd
from itertools import product

import unidecode
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import optimize
from sklearn.linear_model import LinearRegression
import gurobipy as gp
from pulp import *

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# 2026 Season Constants
# ============================================================
NUM_DRIVERS = 22
NUM_TEAMS = 11
SEASON_YEAR = 2026

# Fantasy scoring constants
RACE_POINTS = {1: 25, 2: 18, 3: 15, 4: 12, 5: 10, 6: 8, 7: 6, 8: 4, 9: 2, 10: 1}
FINISH_BONUS = 1          # +1 for classified finish
DNF_PENALTY = -20         # Race DNF
SPRINT_DNF_PENALTY = -10  # Sprint DNF (reduced from -20 in 2026)
FASTEST_LAP_PTS = 5       # Fantasy fastest lap points
Q2_BONUS = 2              # Reaching Q2
Q3_BONUS = 3              # Reaching Q3
BEAT_TEAMMATE_RACE = 3
BEAT_TEAMMATE_QUALI = 2
QUAL_STREAK_PTS = 5       # Q3 streak bonus
RACE_STREAK_PTS = 10      # Top-10 streak bonus
POS_GAIN_PTS = 2          # Per position gained (max +10)

# Budget
FANTASY_BUDGET = 100      # $100M budget cap

# 2026 driver/team mapping
TEAMS_2026 = {
    'McLaren': ['Norris', 'Piastri'],
    'Mercedes': ['Russell', 'Antonelli'],
    'Red Bull': ['Verstappen', 'Hadjar'],
    'Ferrari': ['Leclerc', 'Hamilton'],
    'Williams': ['Albon', 'Sainz'],
    'Racing Bulls': ['Lawson', 'Lindblad'],
    'Aston Martin': ['Alonso', 'Stroll'],
    'Haas': ['Bearman', 'Ocon'],
    'Audi': ['Hulkenberg', 'Bortoleto'],
    'Alpine': ['Gasly', 'Colapinto'],
    'Cadillac': ['Perez', 'Bottas'],
}

# Reverse mapping: driver -> team
DRIVER_TEAM = {}
for team, drivers in TEAMS_2026.items():
    for d in drivers:
        DRIVER_TEAM[d] = team

ALL_DRIVERS = list(DRIVER_TEAM.keys())
ALL_TEAMS = list(TEAMS_2026.keys())
print(f"Grid: {len(ALL_DRIVERS)} drivers across {len(ALL_TEAMS)} teams")

## Part 1: Converting Odds into Implied Probabilities

The first step is to collect betting odds for each driver across 6 markets:
- `winner`: Win probability
- `top3`: Podium probability
- `top6`: Top-6 finish
- `top10`: Points finish
- `flap`: Fastest lap
- `dnf`: Did not finish

We use the shortest decimal odds available (highest implied probability) from spread betting and traditional bookmakers.

In [ ]:
# Load odds data - update odds_2026.csv before each race with latest bookmaker odds
# Format: driver,team,winner,top3,top6,top10,flap,dnf
df = pd.read_csv('odds_2026.csv', index_col=0)

# Verify all 2026 drivers are present
missing = [d for d in ALL_DRIVERS if d not in df.index]
if missing:
    print(f"WARNING: Missing drivers in odds_2026.csv: {missing}")

df

Convert decimal odds to implied probabilities (multiplicative inverse), then correct for bookmaker overround using the power method.

In [ ]:
odds_cols = ['winner', 'top3', 'top6', 'top10', 'flap', 'dnf']
for col in odds_cols:
    df[col] **= -1
df

In [ ]:
def adjust_odds(odds_series, target_sum, method='power'):
    """Remove bookmaker overround using power or linear normalization."""
    if method == 'power':
        def f(x):
            return sum([i**x for i in odds_series]) - target_sum
        k = optimize.fsolve(f, 1)
        return odds_series ** k
    elif method == 'linear':
        return odds_series / sum(odds_series) * target_sum

In [ ]:
# Winner sums to 1, top3 to 3, top6 to 6, top10 to 10, flap to 1
for col, target in zip(odds_cols[:-1], [1, 3, 6, 10, 1]):
    df[col] = adjust_odds(df[col], target)
df

Adjust DNF probabilities so that a driver's DNF + top10 probability cannot exceed 1. We use a log-ratio power transform anchored to the constraint that top drivers have ~96% chance of top-10 if they finish.

In [ ]:
df['dnf'] **= (np.log(0.96 - df['top10']) / np.log(df['dnf'])).max()
df['winner'] = df[['winner', 'top3']].min(axis=1)
df

## Part 2: Estimating Race Finishing Position Distributions

For each driver, we solve a quadratic programming problem (Gurobi) to find the smoothest probability distribution over finishing positions P1-P22, anchored to the odds-implied probabilities for each bracket.

**2026 change**: 22 positions instead of 20 (Cadillac adds 2 drivers).

In [ ]:
def get_probs(idx):
    """Get uniform distribution within each odds bracket for a driver."""
    probs = [df.loc[idx, 'winner']]
    probs += [(df.loc[idx, 'top3'] - df.loc[idx, 'winner']) / 2] * 2
    probs += [(df.loc[idx, 'top6'] - df.loc[idx, 'top3']) / 3] * 3
    probs += [(df.loc[idx, 'top10'] - df.loc[idx, 'top6']) / 4] * 4
    # 12 non-points positions for 22-driver grid (P11-P22)
    probs += [(1 - df.loc[idx, 'top10'] - df.loc[idx, 'dnf']) / 12] * 12
    probs.append(df.loc[idx, 'dnf'])
    return probs

driver = 'Verstappen'
plt.figure(figsize=(12, 4))
plt.bar(['P{}'.format(i) for i in range(1, NUM_DRIVERS + 1)] + ['DNF'], get_probs(driver))
plt.xticks(rotation=90)
plt.ylabel('Probability')
plt.title(f'{driver} - Uniform Position Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate smooth distribution for one driver
quadratic_model = gp.Model('quadratic')
quadratic_model.setParam('outputFlag', False)
driver = 'Verstappen'

# Variables: probability of finishing in each position P1-P22
x = {i: quadratic_model.addVar(vtype=gp.GRB.CONTINUOUS, lb=0, ub=1, name=f'x_{i}')
     for i in range(1, NUM_DRIVERS + 1)}

# Objective: minimize sum of squared differences between consecutive positions
obj = sum([(x[i] - x[i + 1])**2 for i in range(1, NUM_DRIVERS)])
quadratic_model.setObjective(obj, gp.GRB.MINIMIZE)

# Constraints from odds brackets
quadratic_model.addConstr(x[1] == df.loc[driver, 'winner'])
quadratic_model.addConstr(x[2] + x[3] == df.loc[driver, 'top3'] - df.loc[driver, 'winner'])
quadratic_model.addConstr(x[4] + x[5] + x[6] == df.loc[driver, 'top6'] - df.loc[driver, 'top3'])
quadratic_model.addConstr(x[7] + x[8] + x[9] + x[10] == df.loc[driver, 'top10'] - df.loc[driver, 'top6'])
quadratic_model.addConstr(sum([x[i] for i in range(11, NUM_DRIVERS + 1)]) == 1 - df.loc[driver, 'top10'] - df.loc[driver, 'dnf'])
if df.loc[driver, 'top10'] >= 0.5:
    quadratic_model.addConstr(x[NUM_DRIVERS] == 0)

quadratic_model.optimize()
vals = [v.x for v in quadratic_model.getVars()]

plt.figure(figsize=(12, 4))
plt.bar(['P{}'.format(i) for i in range(1, NUM_DRIVERS + 1)] + ['DNF'], vals + [df.loc[driver, 'dnf']])
plt.xticks(rotation=90)
plt.ylabel('Probability')
plt.title(f'{driver} - Smoothed Position Distribution (Gurobi QP)')
plt.tight_layout()
plt.show()

In [ ]:
# Compute smoothed position distributions for all drivers
pos_dict = {}

for driver in df.index:
    quadratic_model = gp.Model('quadratic')
    quadratic_model.setParam('outputFlag', False)
    x = {i: quadratic_model.addVar(vtype=gp.GRB.CONTINUOUS, lb=0, ub=1, name=f'x_{i}')
         for i in range(1, NUM_DRIVERS + 1)}
    obj = sum([(x[i] - x[i + 1])**2 for i in range(1, NUM_DRIVERS)])
    quadratic_model.setObjective(obj, gp.GRB.MINIMIZE)
    quadratic_model.addConstr(x[1] == df.loc[driver, 'winner'])
    quadratic_model.addConstr(x[2] + x[3] == df.loc[driver, 'top3'] - df.loc[driver, 'winner'])
    quadratic_model.addConstr(x[4] + x[5] + x[6] == df.loc[driver, 'top6'] - df.loc[driver, 'top3'])
    quadratic_model.addConstr(x[7] + x[8] + x[9] + x[10] == df.loc[driver, 'top10'] - df.loc[driver, 'top6'])
    quadratic_model.addConstr(sum([x[i] for i in range(11, NUM_DRIVERS + 1)]) == 1 - df.loc[driver, 'top10'] - df.loc[driver, 'dnf'])
    if df.loc[driver, 'top10'] >= 0.5:
        quadratic_model.addConstr(x[NUM_DRIVERS] == 0)
    quadratic_model.optimize()
    pos_dict[driver] = {i + 1: v.x for i, v in enumerate(quadratic_model.getVars())}

print(f"Position distributions computed for {len(pos_dict)} drivers")

In [ ]:
pos_df = pd.DataFrame(pos_dict).T
pos_df['dnf'] = df.dnf
pos_df['team'] = df.team
pos_df['mean_pos'] = sum([i / (1 - df['dnf']) * pos_df[i] for i in range(1, NUM_DRIVERS + 1)])
pos_df.head(10).T

## Part 3: Race EV, Fastest Lap, and DNF Penalty

Compute expected fantasy points from race finishing position, fastest lap probability, and DNF penalty.

In [ ]:
ev_df = pd.DataFrame(index=df.index)

# Race position points: P1=25, P2=18, ..., P10=1, plus +1 finish bonus
ev_df['race'] = (25 * pos_df[1] + 18 * pos_df[2] + 15 * pos_df[3]
                 + 12 * pos_df[4] + 10 * pos_df[5] + 8 * pos_df[6]
                 + 6 * pos_df[7] + 4 * pos_df[8] + 2 * pos_df[9]
                 + pos_df[10] + (1 - pos_df['dnf']) * FINISH_BONUS)

# Fastest lap (5 fantasy points)
ev_df['flap'] = df['flap'] * FASTEST_LAP_PTS

# DNF penalty (-20 points)
ev_df['dnf'] = DNF_PENALTY * df['dnf']

ev_df.sort_values('race', ascending=False).head(10)

## Part 3b: Sprint Race EV

For sprint weekends (6 in 2026), we model additional points from sprint races.
Sprint scoring: P1=8, P2=7, ..., P8=1. Sprint DNF penalty is -10 (reduced from -20 in 2026).

Since sprint results correlate with race pace, we approximate sprint EV from the position distributions.

In [ ]:
# Sprint points: P1=8, P2=7, ..., P8=1
SPRINT_POINTS = {1: 8, 2: 7, 3: 6, 4: 5, 5: 4, 6: 3, 7: 2, 8: 1}
NUM_SPRINTS = 6

# Sprint EV per weekend (approximate from position distribution)
sprint_ev = sum([SPRINT_POINTS.get(i, 0) * pos_df[i] for i in range(1, NUM_DRIVERS + 1)])
sprint_ev += SPRINT_DNF_PENALTY * df['dnf']

# Scale to per-race average (6 sprints out of 24 races)
ev_df['sprint'] = sprint_ev * NUM_SPRINTS / 24

ev_df.sort_values('sprint', ascending=False).head(10)

## Part 4: Modelling Points from Non-Odds Sources

We use historical F1 data (2021-2025) to build regression models predicting:
1. **Qualifying EV** from mean race position (polynomial)
2. **Q2 appearance probability** (logistic regression)
3. **Q3 appearance probability** (logistic regression)
4. **Position gains EV** (linear regression)

**2026 Regulation Reset Note**: The 2022-2025 ground-effect era data is most relevant.
We use all available data but the relationships between grid/race position and fantasy points
should remain stable across regulation changes.

In [ ]:
race_hist_df = pd.read_csv("f1db_csv/races.csv")
hist_result_df = pd.read_csv("f1db_csv/results.csv")

# Use 2021-2025 data (or whatever is available)
max_year = race_hist_df['year'].max()
hist_years = list(range(2021, max_year + 1))
print(f"Using historical data from {hist_years[0]} to {hist_years[-1]}")

In [ ]:
# Gather IDs of all races per year
race_dict = {}
for year in hist_years:
    races = race_hist_df[race_hist_df['year'] == year]['raceId'].tolist()
    race_dict[year] = races

# Collect per-driver season stats
values = {'grid': [], 'position': [], 'fantasy': [], 'top10': [], 'top15': [], 'mean_gain': []}

for year in race_dict:
    season_df = hist_result_df[hist_result_df['raceId'].isin(race_dict[year])]
    drivers = season_df['driverId'].unique()
    for driver in drivers:
        driver_season = season_df[season_df['driverId'] == driver]
        grid_pos_df = driver_season[['grid', 'position']][
            driver_season['position'].str.isnumeric() & (driver_season['grid'] != 0)
        ].copy()
        grid_pos_df['grid'] = grid_pos_df['grid'].astype(float)
        grid_pos_df['position'] = grid_pos_df['position'].astype(float)

        if len(grid_pos_df) > 15:
            values['grid'].append(grid_pos_df['grid'].mean())
            values['position'].append(grid_pos_df['position'].mean())
            values['fantasy'].append(
                sum([sum(grid_pos_df['grid'] == i) * (11 - i) for i in range(1, 11)]) / len(grid_pos_df)
            )
            values['top10'].append(sum(grid_pos_df['grid'] <= 10) / len(grid_pos_df))
            values['top15'].append(sum(grid_pos_df['grid'] <= 15) / len(grid_pos_df))

            total = 0
            for idx in grid_pos_df.index:
                g = grid_pos_df.loc[idx, 'grid']
                p = grid_pos_df.loc[idx, 'position']
                if g - p > 0:
                    total += min(10, (g - p) * POS_GAIN_PTS)
                elif g - p < 0:
                    if g <= 10:
                        total += max(-10, (g - p) * POS_GAIN_PTS)
                    else:
                        total += max(-5, g - p)
            values['mean_gain'].append(total / len(grid_pos_df))

hist_stats_df = pd.DataFrame(values)
print(f"Collected {len(hist_stats_df)} driver-seasons for regression")
hist_stats_df.head()

### Qualifying EV from Mean Race Position

In [ ]:
plt.figure(figsize=(10, 5))
sns.scatterplot(x='position', y='fantasy', data=hist_stats_df)
weights = np.polyfit(hist_stats_df['position'], hist_stats_df['fantasy'], 2)
model_qual = np.poly1d(weights)
x_line = np.linspace(1, 20)
plt.plot(x_line, model_qual(x_line), 'r-', label='Degree-2 polynomial fit')
plt.xlabel('Mean Race Position')
plt.ylabel('Mean Qualifying Fantasy Points')
plt.title('Qualifying EV vs Mean Race Position')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def qual_pts(pos):
    weights = np.polyfit(hist_stats_df['position'], hist_stats_df['fantasy'], 2)
    model = np.poly1d(weights)
    return max(model(pos), 0)

### Q2 Appearance Probability (Logistic Regression)

In [ ]:
hist_stats_df = hist_stats_df.sort_values(by='position')

plt.figure(figsize=(10, 5))
sns.scatterplot(x=hist_stats_df['position'], y=hist_stats_df['top10'])

x_data = hist_stats_df['position'].to_numpy()
y_data = hist_stats_df['top10'].to_numpy()

def logistic_f(x, a, b, c, d):
    return a / (1. + np.exp(-c * (x - d))) + b

popt, pcov = optimize.curve_fit(logistic_f, x_data, y_data, method="trf")
y_fit = logistic_f(x_data, *popt)
plt.plot(x_data, y_fit, 'r-', label='Logistic fit')
plt.xlabel('Mean Race Position')
plt.ylabel('Q2 Appearance Rate')
plt.title('Q2 Appearance Probability')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def q2_finish_pts(pos):
    x = hist_stats_df['position'].to_numpy()
    y = hist_stats_df['top10'].to_numpy()
    def f(x, a, b, c, d):
        return a / (1. + np.exp(-c * (x - d))) + b
    popt, pcov = optimize.curve_fit(f, x, y, method="trf")
    return f(pos, *popt) * Q2_BONUS

### Q3 Appearance Probability

In [ ]:
plt.figure(figsize=(10, 5))
sns.scatterplot(x=hist_stats_df['position'], y=hist_stats_df['top15'])

x_data = hist_stats_df['position'].to_numpy()
y_data = hist_stats_df['top15'].to_numpy()

popt, pcov = optimize.curve_fit(logistic_f, x_data, y_data, method="trf")
y_fit = logistic_f(x_data, *popt)
plt.plot(x_data, y_fit, 'r-', label='Logistic fit')
plt.xlabel('Mean Race Position')
plt.ylabel('Q3 Appearance Rate')
plt.title('Q3 Appearance Probability')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def q3_finish_pts(pos):
    x = hist_stats_df['position'].to_numpy()
    y = hist_stats_df['top15'].to_numpy()
    def f(x, a, b, c, d):
        return a / (1. + np.exp(-c * (x - d))) + b
    popt, pcov = optimize.curve_fit(f, x, y, method="trf")
    return max(f(pos, *popt) * Q3_BONUS, 0)

### Position Gains EV

In [ ]:
plt.figure(figsize=(10, 5))
sns.lmplot(x='position', y='mean_gain', data=hist_stats_df)
plt.title('Position Gain EV vs Mean Race Position')
plt.tight_layout()
plt.show()

In [ ]:
def gain_pts(pos):
    x = hist_stats_df['position'].values.reshape(-1, 1)
    y = hist_stats_df['mean_gain'].values.reshape(-1, 1)
    reg = LinearRegression().fit(x, y)
    return reg.predict(np.array(pos).reshape(-1, 1))[0][0]

### Apply Regression Models to 2026 Drivers

In [ ]:
ev_df['qual'] = [qual_pts(i) for i in pos_df['mean_pos']]
ev_df['gain'] = [gain_pts(i) for i in pos_df['mean_pos']]
ev_df['q'] = [q2_finish_pts(i) + q3_finish_pts(i) for i in pos_df['mean_pos']]

ev_df.sort_values('qual', ascending=False).head(10)

## Part 5: Teammate Comparisons

We scrape current-season results to estimate the probability of each driver beating their teammate.
Probabilities are Bayesian-shrunk toward 0.5 to account for small sample sizes.

Since the 2026 season hasn't started yet (or has just begun), we use pre-season estimates
based on relative expected positions from odds, or early-season data as available.

In [ ]:
# 2026 race URLs - update this list as races complete
# Format: "raceId/circuit-name"
hrefs = [
    # "XXXX/australia",  # Round 1 - Melbourne
    # Add race URLs as the season progresses
]

# https://www.formula1.com/en/results.html/2026/races/XXXX/australia/qualifying.html
qualifying_dfs, race_dfs, grid_dfs = [], [], []
season_length = 0  # Update after each race

for href in hrefs[:season_length]:
    qualifying_url = f'https://www.formula1.com/en/results.html/2026/races/{href}/qualifying.html'
    qual_result = pd.read_html(qualifying_url)[0]
    qual_result['Driver'] = [unidecode.unidecode(name.split('  ')[1]) for name in qual_result['Driver']]
    qualifying_dfs.append(qual_result.set_index('Driver')[['Pos', 'Car', 'Q2', 'Q3']])

    race_url = f'https://www.formula1.com/en/results.html/2026/races/{href}.html'
    race_result = pd.read_html(race_url)[0]
    race_result['Driver'] = [unidecode.unidecode(name.split('  ')[1]) for name in race_result['Driver']]
    race_dfs.append(race_result.set_index('Driver')[['Pos', 'Car', 'Time/Retired']])

    grid_url = f'https://www.formula1.com/en/results.html/2026/races/{href}/starting-grid.html'
    grid_result = pd.read_html(grid_url)[0]
    grid_result['Driver'] = [unidecode.unidecode(name.split('  ')[1]) for name in grid_result['Driver']]
    grid_dfs.append(grid_result.set_index('Driver')[['Pos', 'Car']])

print(f"Loaded {season_length} race(s) of 2026 season data")

In [ ]:
if season_length > 0 and len(race_dfs) > 0:
    # Use actual season data
    drivers_list = race_dfs[0].index
    event_df = pd.DataFrame(index=drivers_list)
    event_df['team'] = race_dfs[0].Car
    event_df['beat_teammate_race'] = np.empty((len(drivers_list), 0)).tolist()
    event_df['beat_teammate_qualifying'] = np.empty((len(drivers_list), 0)).tolist()

    for i, (r_df, q_df, g_df) in enumerate(zip(race_dfs, qualifying_dfs, grid_dfs)):
        for team_name in r_df['Car'].unique():
            team_r = r_df.loc[r_df['Car'] == team_name]
            try:
                event_df.loc[team_r.index[0], 'beat_teammate_race'].append(1)
                event_df.loc[team_r.index[1], 'beat_teammate_race'].append(0)
            except (IndexError, KeyError):
                continue
            team_q = q_df.loc[q_df['Car'] == team_name]
            try:
                event_df.loc[team_q.index[0], 'beat_teammate_qualifying'].append(1)
                event_df.loc[team_q.index[1], 'beat_teammate_qualifying'].append(0)
            except (IndexError, KeyError):
                continue

    # Bayesian shrinkage toward 0.5
    event_df['P(beat_teammate_race)'] = [np.mean(l) if l else 0.5 for l in event_df['beat_teammate_race']]
    event_df['P(beat_teammate_race)'] = (5 * event_df['P(beat_teammate_race)'] + 0.5) / 6
    event_df['P(beat_teammate_qualifying)'] = [np.mean(l) if l else 0.5 for l in event_df['beat_teammate_qualifying']]
    event_df['P(beat_teammate_qualifying)'] = (5 * event_df['P(beat_teammate_qualifying)'] + 0.5) / 6
    event_df = pd.DataFrame(event_df[['team', 'P(beat_teammate_race)', 'P(beat_teammate_qualifying)']])
else:
    # Pre-season estimates based on driver strength priors from odds
    event_df = pd.DataFrame(index=df.index)
    event_df['team'] = df['team']

    for team_name, team_drivers in TEAMS_2026.items():
        if all(d in pos_df.index for d in team_drivers):
            d1, d2 = team_drivers
            mp1 = pos_df.loc[d1, 'mean_pos']
            mp2 = pos_df.loc[d2, 'mean_pos']
            # Convert to win probability using logistic function
            diff = mp2 - mp1  # positive if d1 is stronger
            p1 = 1 / (1 + np.exp(-0.3 * diff))
            p1 = (5 * p1 + 0.5) / 6  # shrink toward 0.5
            event_df.loc[d1, 'P(beat_teammate_race)'] = p1
            event_df.loc[d2, 'P(beat_teammate_race)'] = 1 - p1
            event_df.loc[d1, 'P(beat_teammate_qualifying)'] = p1
            event_df.loc[d2, 'P(beat_teammate_qualifying)'] = 1 - p1

print("Teammate beat probabilities:")
event_df[['team', 'P(beat_teammate_race)', 'P(beat_teammate_qualifying)']].sort_values('P(beat_teammate_race)', ascending=False)

In [ ]:
# Adjust teammate probabilities for DNF scenarios
for team_name in event_df['team'].unique():
    team_drivers = event_df[event_df['team'] == team_name].index.tolist()
    if len(team_drivers) != 2:
        continue
    d1, d2 = team_drivers[0], team_drivers[1]
    if d1 not in df.index or d2 not in df.index:
        continue

    p_no_dnf = (1 - df.loc[d1, 'dnf']) * (1 - df.loc[d2, 'dnf'])
    adjusted = adjust_odds(
        np.array(event_df.loc[[d1, d2], 'P(beat_teammate_race)']),
        p_no_dnf, method='linear'
    )
    event_df.loc[[d1, d2], 'P(beat_teammate_race)'] = adjusted

    # Add unilateral DNF scenarios
    event_df.loc[d1, 'P(beat_teammate_race)'] += (1 - df.loc[d1, 'dnf']) * df.loc[d2, 'dnf']
    event_df.loc[d2, 'P(beat_teammate_race)'] += (1 - df.loc[d2, 'dnf']) * df.loc[d1, 'dnf']

event_df

In [ ]:
for driver in df.index:
    if driver in event_df.index:
        ev_df.loc[driver, 'beat_teammate'] = (
            event_df.loc[driver, 'P(beat_teammate_race)'] * BEAT_TEAMMATE_RACE
            + event_df.loc[driver, 'P(beat_teammate_qualifying)'] * BEAT_TEAMMATE_QUALI
        )
    else:
        print(f"Warning: {driver} not in event_df")
        ev_df.loc[driver, 'beat_teammate'] = 0

## Part 6: Streak Modelling

Drivers earn bonus points for consecutive qualifying/race streaks:
- **Qualifying streak**: Q3 in consecutive races -> +5 points per race
- **Race streak**: Top-10 in consecutive races -> +10 points per race

Update these lists based on current streak status:

In [ ]:
# UPDATE: Drivers currently on streaks (empty at season start)
q_streak_drivers = []    # Drivers on Q3 qualifying streaks
race_streak_drivers = [] # Drivers on top-10 race streaks

In [ ]:
for driver in pos_df.index:
    if driver in q_streak_drivers:
        streak_prob = q3_finish_pts(pos_df.loc[driver, 'mean_pos']) / Q3_BONUS
        ev_df.loc[driver, 'q_streak'] = streak_prob * QUAL_STREAK_PTS
    else:
        ev_df.loc[driver, 'q_streak'] = 0

for driver in pos_df.index:
    if driver in race_streak_drivers:
        streak_prob = sum([pos_df.loc[driver, i] for i in range(1, 11)])
        ev_df.loc[driver, 'race_streak'] = streak_prob * RACE_STREAK_PTS
    else:
        ev_df.loc[driver, 'race_streak'] = 0

ev_df

## Part 7: Final EV Compilation

In [ ]:
driver_df = pd.DataFrame(ev_df.T.sum(), columns=['EV'])

print("=== Driver EV Rankings ===")
driver_df.sort_values(by='EV', ascending=False)

In [ ]:
team_df = pd.DataFrame(index=pos_df['team'].unique().tolist())

for team in pos_df['team'].unique():
    t_df = ev_df[pos_df['team'] == team]
    team_df.loc[team, 'EV'] = (
        t_df['race'].sum() + t_df['qual'].sum() + t_df['q'].sum()
        + t_df['gain'].sum() + t_df['sprint'].sum()
    )

print("=== Constructor EV Rankings ===")
team_df.sort_values(by='EV', ascending=False)

In [ ]:
# UPDATE: Teams currently on streaks
q_streak_teams = []
race_streak_teams = []

for team in q_streak_teams:
    drivers_t = pos_df[pos_df['team'] == team].index.tolist()
    streak_prob = (q3_finish_pts(pos_df.loc[drivers_t[0], 'mean_pos']) / Q3_BONUS
                   * q3_finish_pts(pos_df.loc[drivers_t[1], 'mean_pos']) / Q3_BONUS)
    team_df.loc[team, 'EV'] += streak_prob * QUAL_STREAK_PTS

for team in race_streak_teams:
    drivers_t = pos_df[pos_df['team'] == team].index.tolist()
    streak_prob = (sum([pos_df.loc[drivers_t[0], i] for i in range(1, 11)])
                   * sum([pos_df.loc[drivers_t[1], i] for i in range(1, 11)]))
    team_df.loc[team, 'EV'] += streak_prob * RACE_STREAK_PTS

team_df.sort_values(by='EV', ascending=False)

## Part 8: Team Optimization (PuLP Integer LP)

Using the EV projections, solve for the optimal Fantasy team:
- **5 drivers + 2 constructors**
- **Budget**: $100M
- **1 turbo driver** (scores double)
- Subject to transfer constraints from current team

In [ ]:
# Load prices - UPDATE with 2026 prices source
try:
    prices_pd = pd.read_csv('https://docs.google.com/spreadsheets/d/1b5486fzknvQgXJxPuS7U88HwMYLrz-EfdfTO3-w5qx0/pub?output=csv')
except:
    prices_pd = pd.read_csv('_prices.csv')

known_drivers_2026 = ALL_DRIVERS
known_teams_2026 = ALL_TEAMS

# Extract prices
driver_prices_raw = dict(zip(prices_pd['Name'], prices_pd['CurrentPrice']))
driver_prices = {k: float(v) for k, v in driver_prices_raw.items() if k in known_drivers_2026}
team_prices = {k: float(v) for k, v in driver_prices_raw.items() if k in known_teams_2026}

# Handle name variations
name_fixes = {
    'Kick Sauber': 'Audi',
    'Mclaren': 'McLaren',
    'RB': 'Racing Bulls',
}
for old, new in name_fixes.items():
    if old in team_prices and new not in team_prices:
        team_prices[new] = team_prices.pop(old)

# Placeholder prices for new/missing entries
for d in known_drivers_2026:
    if d not in driver_prices:
        driver_prices[d] = 5.0
        print(f"  Placeholder price for {d}: $5.0M")
for t in known_teams_2026:
    if t not in team_prices:
        team_prices[t] = 5.0
        print(f"  Placeholder price for {t}: $5.0M")

for d in driver_prices:
    if d in driver_df.index:
        driver_df.loc[d, 'price'] = driver_prices[d]
for t in team_prices:
    if t in team_df.index:
        team_df.loc[t, 'price'] = team_prices[t]

print("\nDriver Prices:")
print(driver_df.sort_values('EV', ascending=False))
print("\nTeam Prices:")
print(team_df.sort_values('EV', ascending=False))

In [ ]:
# Value analysis: EV per $M
driver_df['EV/$M'] = driver_df['EV'] / driver_df['price']
team_df['EV/$M'] = team_df['EV'] / team_df['price']

print("=== Best Value Drivers (EV/$M) ===")
display(driver_df.sort_values('EV/$M', ascending=False))
print("\n=== Best Value Teams (EV/$M) ===")
display(team_df.sort_values('EV/$M', ascending=False))

In [ ]:
pd.concat([
    team_df.sort_values(by='EV', ascending=False),
    driver_df.sort_values(by='EV', ascending=False)
])

In [ ]:
def solve_team(budget, owned_drivers, owned_team, force_include, force_exclude, transfers=3):
    """
    Solve for optimal F1 Fantasy team using integer linear programming.
    
    Args:
        budget: Total budget in $M
        owned_drivers: List of currently owned driver names
        owned_team: List of currently owned team names
        force_include: Dict with 'driver' and 'team' lists to force into solution
        force_exclude: Dict with 'driver' and 'team' lists to exclude
        transfers: Number of available transfers
    
    Returns:
        Dict with optimal drivers, teams, turbo driver, and total EV
    """
    owned_d = {d: int(d in owned_drivers) for d in driver_df.index}
    owned_t = {c: int(c in owned_team) for c in team_df.index}

    prob = LpProblem("F1_Fantasy_2026", LpMaximize)
    drivers = LpVariable.dicts('drivers', driver_df.index, cat='Binary')
    teams = LpVariable.dicts('teams', team_df.index, cat='Binary')
    turbo = LpVariable.dicts('turbo', driver_df.index, cat='Binary')

    # Objective: maximize total EV (turbo driver scores double)
    prob += (lpSum([drivers[d] * driver_df.loc[d, 'EV'] for d in drivers])
             + lpSum([teams[t] * team_df.loc[t, 'EV'] for t in teams])
             + lpSum([turbo[d] * driver_df.loc[d, 'EV'] for d in drivers]))

    # Constraints
    prob += lpSum(drivers) == 5           # Exactly 5 drivers
    prob += lpSum(teams) == 2             # Exactly 2 constructors
    prob += lpSum(turbo) == 1             # Exactly 1 turbo driver

    # Budget constraint
    prob += (lpSum([drivers[d] * driver_df.loc[d, 'price'] for d in drivers])
             + lpSum([teams[t] * team_df.loc[t, 'price'] for t in teams]) <= budget)

    # Turbo driver must be selected
    for d in drivers:
        prob += turbo[d] <= drivers[d]

    # Transfer constraint
    prob += (lpSum([owned_d[d] * drivers[d] for d in drivers])
             + lpSum([owned_t[t] * teams[t] for t in teams]) >= 7 - transfers)

    # Force include/exclude
    for d in force_include.get('driver', []):
        if d in drivers:
            prob += drivers[d] == 1
    for t in force_include.get('team', []):
        if t in teams:
            prob += teams[t] == 1
    for d in force_exclude.get('driver', []):
        if d in drivers:
            prob += drivers[d] == 0
    for t in force_exclude.get('team', []):
        if t in teams:
            prob += teams[t] == 0

    prob.solve(PULP_CBC_CMD(msg=0))

    return {
        'drivers': sorted([d for d in drivers if drivers[d].varValue == 1],
                          key=lambda x: -driver_prices.get(x, 0)),
        'team': sorted([t for t in teams if teams[t].varValue == 1],
                       key=lambda x: -team_prices.get(x, 0)),
        'turbo driver': [d for d in drivers if turbo[d].varValue == 1][0],
        'EV': value(prob.objective)
    }

### Optimal Team Selection

In [ ]:
# ============================================================
# UPDATE: Your current team and constraints
# ============================================================

# Fresh start - optimal team with no constraints
print("=== Optimal Team (Fresh Start) ===")
print(solve_team(
    budget=FANTASY_BUDGET,
    owned_drivers=[],
    owned_team=[],
    force_include={'driver': [], 'team': []},
    force_exclude={'driver': [], 'team': []},
    transfers=12
))

In [ ]:
# Optimal teams across budget range
optimal_teams = {}
min_budget, max_budget = 95, FANTASY_BUDGET
last_team = {}
b = min_budget

while b <= max_budget:
    b = round(b, 1)
    team = solve_team(
        budget=b,
        owned_drivers=[],
        owned_team=[],
        force_include={'driver': [], 'team': []},
        force_exclude={'driver': [], 'team': []},
        transfers=12
    )
    if team != last_team:
        optimal_teams[b] = team
    last_team = team
    b += 0.1

out_df = pd.DataFrame(optimal_teams).T
for i in range(5):
    out_df[f'driver {i+1}'] = [d[i] for d in out_df['drivers']]
out_df.index = [f'{out_df.index[i-1]}-{round(out_df.index[i] - 0.1, 1)}'
                for i in range(1, len(out_df))] + [f'{out_df.index[-1]}-{max_budget}']
out_df.index.name = 'budget range'
out_df.drop('drivers', axis=1)[['driver 1', 'driver 2', 'driver 3', 'driver 4', 'driver 5',
                                 'turbo driver', 'team', 'EV']]

## Part 9: EV Breakdown Visualization

In [ ]:
# Stacked bar chart of EV components per driver
fig, ax = plt.subplots(figsize=(16, 8))

ev_components = ['race', 'flap', 'dnf', 'sprint', 'qual', 'gain', 'q', 'beat_teammate', 'q_streak', 'race_streak']
available_components = [c for c in ev_components if c in ev_df.columns]

sorted_drivers = ev_df[available_components].sum(axis=1).sort_values(ascending=True).index

bottom = np.zeros(len(sorted_drivers))
colors = plt.cm.Set3(np.linspace(0, 1, len(available_components)))

for i, component in enumerate(available_components):
    values = ev_df.loc[sorted_drivers, component].values
    ax.barh(range(len(sorted_drivers)), values, left=bottom, label=component, color=colors[i])
    bottom += values

ax.set_yticks(range(len(sorted_drivers)))
ax.set_yticklabels(sorted_drivers)
ax.set_xlabel('Expected Fantasy Points')
ax.set_title('F1 Fantasy 2026 - EV Breakdown by Driver')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Notes on 2026 Regulation Reset

**Key uncertainties for 2026 predictions:**

1. **New regulation era**: The 2026 rules represent a fundamental reset. Historical performance from 2022-2025 is less predictive than usual. The odds markets (which incorporate pre-season testing and expert analysis) are our best signal.

2. **Power unit maturity**: New PU formula means manufacturers start from scratch. Teams with better electrical systems (MGU-K tripled to 350kW) will have structural advantages. Mercedes supplies 4 teams, Ferrari 3, RBPT 2, Honda 1, Audi 1.

3. **Active aero dynamics**: DRS is gone. Overtaking now depends on electrical energy management (Boost/Overtake modes). This could change the typical grid-to-race-position relationship.

4. **New teams**: Cadillac (11th team) and Audi (rebranded Sauber) add significant uncertainty. Both are expected to be at the back initially.

5. **Pre-season testing signals** (Bahrain 2026):
   - Ferrari (Leclerc): Fastest overall - 1:31.992
   - Mercedes (Antonelli): 2nd - 1:32.803
   - McLaren (Piastri/Norris): 3rd/4th - within 0.1s of each other
   - Red Bull (Verstappen): 5th - 1:33.109

**Model accuracy**: As with 2024, formal backtesting is not possible. The model's value lies in systematically converting market odds into optimal Fantasy team selections, not in predicting individual race outcomes.